# 🧬 Biotech Clinical Trial Event Study
## Project #5 — Drug Approval Returns Analysis
### Healthcare Equity Research | Google Colab Starter Notebook

---

**Ten cells. Run them in order.**

| Cell | Module | Output |
|------|--------|--------|
| 1 | Install & imports | Environment ready |
| 2 | Biotech universe & config | 45-company watchlist |
| 3 | SEC EDGAR 8-K scraper | Raw clinical event filings |
| 4 | ClinicalTrials.gov pipeline | Live trial calendar |
| 5 | Event classification engine | Labeled event database |
| 6 | Price data & abnormal returns | CAR computation |
| 7 | Event study analysis | Core results |
| 8 | Phase 2→3 pricing analysis | Signal decay study |
| 9 | Upcoming catalyst dashboard | Forward-looking calendar |

> ⚠️ **Run Cell 1 first, restart runtime, then run Cells 2–9 in order.**


## Cell 1 — Install dependencies
*Run once. Restart runtime after.*

In [ ]:
# ============================================================
# CELL 1 — Install & environment setup
# ============================================================
!pip install yfinance pandas numpy scipy matplotlib seaborn requests beautifulsoup4 tqdm reportlab lxml statsmodels python-dateutil --quiet

print("✅  Packages installed.")
print("⚠️  RESTART RUNTIME NOW (Runtime → Restart runtime), then run Cell 2+")


## Cell 2 — Universe, config & imports
*Defines the 45-company biotech watchlist and all study parameters.*

In [ ]:
# ============================================================
# CELL 2 — Imports, universe, and study configuration
# ============================================================
import warnings, time, re, json, sqlite3, os
from datetime import datetime, timedelta
from pathlib import Path
from dateutil.parser import parse as dateparse

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.3f}".format)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "normal",
})

# ── Color palette ─────────────────────────────────────────────
C = {
    "success":  "#1D9E75",   # Phase 3 success / FDA approval
    "failure":  "#E24B4A",   # Phase failure / CRL
    "phase2":   "#185FA5",   # Phase 2 events
    "phase3":   "#7F77DD",   # Phase 3 events
    "fda":      "#BA7517",   # FDA decisions
    "neutral":  "#888780",
    "warning":  "#BA7517",
    "light":    "#F1EFE8",
    "bench":    "#5F5E5A",
}

# ── Output directory ──────────────────────────────────────────
DATA_DIR = Path("biotech_event_study")
DATA_DIR.mkdir(exist_ok=True)

# ── Study configuration ───────────────────────────────────────
CFG = {
    # Event study windows (trading days)
    "pre_window":   60,    # days before event to observe
    "post_window":  60,    # days after event to observe
    "estimation_start": -252,   # estimation window start (1 year before event)
    "estimation_end":   -30,    # estimation window end (30 days before event)

    # Date range for 8-K scraping
    "scrape_start": "2019-01-01",
    "scrape_end":   datetime.today().strftime("%Y-%m-%d"),

    # Benchmark for abnormal return computation
    "benchmark": "XBI",    # SPDR S&P Biotech ETF — sector benchmark

    # Minimum liquidity filter
    "min_market_cap_M": 200,   # $200M minimum

    # ClinicalTrials.gov — upcoming events window
    "upcoming_days": 365,
}

# ── Biotech universe ─────────────────────────────────────────
# Selected for: active clinical pipelines, sufficient 8-K history,
# meaningful clinical events over the study period
UNIVERSE = {
    # Large-cap — established commercial + pipeline
    "VRTX":  "Vertex Pharmaceuticals",
    "REGN":  "Regeneron Pharmaceuticals",
    "ALNY":  "Alnylam Pharmaceuticals",
    "BIIB":  "Biogen",
    "BMRN":  "BioMarin Pharmaceutical",
    "SGEN":  "Seagen",          # acquired 2023 — historical data still valid
    "MRNA":  "Moderna",
    "BNTX":  "BioNTech",
    "EXEL":  "Exelixis",
    "INCY":  "Incyte Corporation",

    # Mid-cap — high clinical event frequency
    "RCUS":  "Arcus Biosciences",
    "FOLD":  "Amicus Therapeutics",
    "PTCT":  "PTC Therapeutics",
    "RARE":  "Ultragenyx Pharmaceutical",
    "ACAD":  "ACADIA Pharmaceuticals",
    "IONS":  "Ionis Pharmaceuticals",
    "SRPT":  "Sarepta Therapeutics",
    "BLUE":  "bluebird bio",
    "CRSP":  "CRISPR Therapeutics",
    "BEAM":  "Beam Therapeutics",

    # Small/mid-cap — higher volatility on binary events
    "TRVN":  "Trevena",
    "ATRA":  "Atara Biotherapeutics",
    "KYMR":  "Kymera Therapeutics",
    "VCEL":  "Vericel Corporation",
    "RXRX":  "Recursion Pharmaceuticals",
    "ARQT":  "Arcutis Biotherapeutics",
    "KRTX":  "Karuna Therapeutics",
    "SAGE":  "Sage Therapeutics",
    "RYTM":  "Rhythm Pharmaceuticals",
    "NTLA":  "Intellia Therapeutics",

    # Additional coverage for pipeline depth
    "GILD":  "Gilead Sciences",
    "AMGN":  "Amgen",
    "ABBV":  "AbbVie",
    "MRK":   "Merck",
    "PFE":   "Pfizer",
    "RGEN":  "Repligen Corporation",
    "TECH":  "Bio-Techne",
    "NVAX":  "Novavax",
    "OCGN":  "Ocugen",
    "DVAX":  "Dynavax Technologies",
    "PRTA":  "Prothena Corporation",
    "ZYME":  "Zymeworks",
    "IMVT":  "Immunovant",
    "ARDX":  "Ardelyx",
    "CRVS":  "Corvus Pharmaceuticals",
}

print(f"✅  Study configuration loaded")
print(f"   Universe      : {len(UNIVERSE)} biotech companies")
print(f"   Benchmark     : {CFG['benchmark']} (SPDR S&P Biotech ETF)")
print(f"   Event window  : [{-CFG['pre_window']}, +{CFG['post_window']}] trading days")
print(f"   Study period  : {CFG['scrape_start']} → {CFG['scrape_end']}")
print(f"   Output dir    : {DATA_DIR}/")


## Cell 3 — SEC EDGAR 8-K Clinical Event Scraper
*Pulls 8-K filings from EDGAR and classifies clinical trial announcements. This is real financial data engineering — the same process a healthcare research analyst would do manually.*

In [ ]:
# ============================================================
# CELL 3 — SEC EDGAR 8-K scraper & clinical event classifier
# ============================================================

EDGAR_HEADERS = {
    "User-Agent": "Biotech Event Study Research contact@research.edu",
    "Accept-Encoding": "gzip, deflate",
}
EDGAR_BASE = "https://data.sec.gov"

# ── Keyword classifier ────────────────────────────────────────
# Maps headline keywords → event type
# Ordered: most specific first
EVENT_KEYWORDS = {
    # FDA decisions
    "fda_approval": [
        "fda approves", "fda approved", "approved by the fda",
        "receives fda approval", "granted approval", "nda approved",
        "bla approved", "sNDA approved", "priority review designation",
        "approved for", "receives approval from",
    ],
    "fda_rejection": [
        "complete response letter", "crl", "refused to file",
        "fda issues complete response", "not approvable",
        "rejects", "fda declines",
    ],
    "fda_filing": [
        "nda submission", "bla submission", "sNDA submission",
        "files nda", "files bla", "submits nda", "submits bla",
        "regulatory submission", "pdufa date",
    ],
    "breakthrough_designation": [
        "breakthrough therapy designation", "breakthrough designation",
        "fast track designation", "orphan drug designation",
        "accelerated approval",
    ],
    # Phase 3
    "phase3_success": [
        "phase 3 met", "phase iii met", "phase 3 positive",
        "phase iii positive", "met primary endpoint", "achieved primary endpoint",
        "phase 3 results", "statistically significant", "topline phase 3",
        "pivotal trial met", "phase 3 data", "phase iii data positive",
    ],
    "phase3_failure": [
        "phase 3 did not meet", "phase iii did not meet",
        "failed to meet primary", "phase 3 failed", "phase iii failed",
        "did not achieve primary endpoint", "phase 3 negative",
        "phase 3 did not achieve", "topline phase 3 negative",
    ],
    # Phase 2
    "phase2_success": [
        "phase 2 met", "phase ii met", "phase 2 positive",
        "phase ii positive", "phase 2 results", "phase ii results",
        "phase 2 data", "phase ii data positive", "proof of concept",
        "phase 2 achieved", "positive phase 2",
    ],
    "phase2_failure": [
        "phase 2 did not meet", "phase ii did not meet",
        "phase 2 failed", "phase ii failed", "phase 2 negative",
        "phase ii negative", "did not meet phase 2",
    ],
    # Phase 1
    "phase1_data": [
        "phase 1 data", "phase i data", "phase 1 results",
        "first-in-human", "dose escalation results", "safety data",
        "phase 1 cohort",
    ],
}

# Outcome polarity
EVENT_POLARITY = {
    "fda_approval":           "positive",
    "fda_rejection":          "negative",
    "fda_filing":             "positive",
    "breakthrough_designation":"positive",
    "phase3_success":         "positive",
    "phase3_failure":         "negative",
    "phase2_success":         "positive",
    "phase2_failure":         "negative",
    "phase1_data":            "neutral",
}

EVENT_STAGE = {
    "fda_approval":           "FDA",
    "fda_rejection":          "FDA",
    "fda_filing":             "FDA",
    "breakthrough_designation":"FDA",
    "phase3_success":         "Phase 3",
    "phase3_failure":         "Phase 3",
    "phase2_success":         "Phase 2",
    "phase2_failure":         "Phase 2",
    "phase1_data":            "Phase 1",
}


class EdgarScraper:
    """
    Pulls 8-K filings from SEC EDGAR for a given company CIK.
    Extracts: filing date, headline text, and classifies the clinical event.

    SEC EDGAR provides free, public access to all filings via their
    REST API: https://data.sec.gov/submissions/{CIK}.json
    """

    def __init__(self, headers=EDGAR_HEADERS):
        self.headers = headers
        self.session = requests.Session()
        self.session.headers.update(headers)

    def get_cik(self, ticker: str) -> str | None:
        """Resolve ticker → CIK using EDGAR company tickers JSON."""
        url = f"{EDGAR_BASE}/files/company_tickers.json"
        try:
            r = self.session.get(url, timeout=10)
            data = r.json()
            for entry in data.values():
                if entry.get("ticker", "").upper() == ticker.upper():
                    return str(entry["cik_str"]).zfill(10)
        except Exception:
            pass
        return None

    def get_8k_filings(self, cik: str, start_date: str, end_date: str) -> list[dict]:
        """
        Returns list of 8-K filings with: date, accession number, primary doc URL.
        Uses EDGAR submissions JSON endpoint — no scraping needed.
        """
        url = f"{EDGAR_BASE}/submissions/CIK{cik}.json"
        try:
            r = self.session.get(url, timeout=15)
            sub = r.json()
        except Exception as e:
            return []

        filings = sub.get("filings", {}).get("recent", {})
        if not filings:
            return []

        forms     = filings.get("form", [])
        dates     = filings.get("filingDate", [])
        accessions= filings.get("accessionNumber", [])
        documents = filings.get("primaryDocument", [])

        results = []
        sd, ed = pd.Timestamp(start_date), pd.Timestamp(end_date)

        for form, date, acc, doc in zip(forms, dates, accessions, documents):
            if form not in ("8-K", "8-K/A"):
                continue
            try:
                dt = pd.Timestamp(date)
            except Exception:
                continue
            if not (sd <= dt <= ed):
                continue

            acc_fmt = acc.replace("-", "")
            doc_url = (f"https://www.sec.gov/Archives/edgar/data/"
                       f"{int(cik)}/{acc_fmt}/{doc}")
            results.append({
                "date":       dt,
                "accession":  acc,
                "doc_url":    doc_url,
                "form":       form,
            })

        return results

    def extract_headline(self, doc_url: str, timeout: int = 8) -> str:
        """
        Fetches the 8-K HTML and extracts the first substantive paragraph
        (the press release headline / opening sentence).
        Caps at 500 chars to avoid parsing entire filing.
        """
        try:
            r = self.session.get(doc_url, timeout=timeout)
            soup = BeautifulSoup(r.text, "lxml")
            # Remove script/style noise
            for tag in soup(["script","style","header","footer"]):
                tag.decompose()
            text = soup.get_text(" ", strip=True)
            # Take first 800 chars — headline is always at the top
            return text[:800].strip()
        except Exception:
            return ""

    @staticmethod
    def classify_event(text: str) -> str | None:
        """Returns event type if text matches any keyword group, else None."""
        t = text.lower()
        for event_type, keywords in EVENT_KEYWORDS.items():
            if any(kw in t for kw in keywords):
                return event_type
        return None


# ── Run scraper ───────────────────────────────────────────────
print("🔍  SEC EDGAR 8-K Scraper")
print(f"    Scraping {len(UNIVERSE)} companies from {CFG['scrape_start']} → {CFG['scrape_end']}")
print("    Runtime: ~5–12 min (respectful rate-limiting)\n")

scraper    = EdgarScraper()
all_events = []

for ticker, company in tqdm(UNIVERSE.items(), desc="  Companies", ncols=72):
    # 1. Resolve CIK
    cik = scraper.get_cik(ticker)
    if cik is None:
        continue
    time.sleep(0.12)   # EDGAR rate limit: 10 req/sec

    # 2. Get 8-K list
    filings = scraper.get_8k_filings(cik, CFG["scrape_start"], CFG["scrape_end"])
    time.sleep(0.12)

    # 3. Classify each filing
    for f in filings:
        # First try to classify from filename/URL — saves a full HTML fetch
        # If ambiguous, fetch first 800 chars of the document
        headline = scraper.extract_headline(f["doc_url"])
        time.sleep(0.08)

        event_type = scraper.classify_event(headline)
        if event_type is None:
            continue

        all_events.append({
            "ticker":      ticker,
            "company":     company,
            "date":        f["date"],
            "event_type":  event_type,
            "stage":       EVENT_STAGE[event_type],
            "polarity":    EVENT_POLARITY[event_type],
            "headline":    headline[:300],
            "doc_url":     f["doc_url"],
            "cik":         cik,
        })

events_df = pd.DataFrame(all_events)
if not events_df.empty:
    events_df["date"] = pd.to_datetime(events_df["date"])
    events_df = events_df.sort_values("date").reset_index(drop=True)

    # Save to disk
    events_df.to_parquet(DATA_DIR / "raw_events.parquet", index=False)
    events_df.to_csv(DATA_DIR  / "raw_events.csv",     index=False)

# ── Summary ───────────────────────────────────────────────────
n_events = len(events_df) if not events_df.empty else 0
print(f"\n✅  Scraping complete")
print(f"   Total clinical events found : {n_events}")
if n_events > 0:
    print(f"   Date range                  : {events_df['date'].min().date()} → {events_df['date'].max().date()}")
    print("\n   Breakdown by event type:")
    breakdown = events_df.groupby(["stage","polarity"]).size().rename("count")
    print(breakdown.to_string())
    print("\n   Sample events:")
    display(events_df[["date","ticker","event_type","polarity","headline"]]
            .head(8).style.set_properties(**{"font-size":"11px"}))


## Cell 4 — ClinicalTrials.gov Pipeline Tracker
*Pulls active and upcoming trials for each company in our universe using the free ClinicalTrials.gov API. Builds the forward-looking event calendar.*

In [ ]:
# ============================================================
# CELL 4 — ClinicalTrials.gov pipeline tracker
# ============================================================

class ClinicalTrialsAPI:
    """
    Queries the ClinicalTrials.gov v2 API (free, no key required).
    Documentation: https://clinicaltrials.gov/data-api/api

    Returns structured trial data:
      - NCT ID, title, phase, status, sponsor, conditions, primary completion date
    """

    BASE = "https://clinicaltrials.gov/api/v2/studies"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers["User-Agent"] = "Biotech Research Event Study"

    def search_by_sponsor(self, sponsor_name: str, max_results: int = 50) -> list[dict]:
        """
        Search trials by lead sponsor name.
        Returns trials in Phase 2+ with recruiting or active status.
        """
        params = {
            "query.lead": sponsor_name,
            "filter.overallStatus": "RECRUITING,ACTIVE_NOT_RECRUITING,COMPLETED",
            "fields": "NCTId,BriefTitle,Phase,OverallStatus,LeadSponsorName,"
                      "Condition,PrimaryCompletionDate,StartDate,EnrollmentCount",
            "pageSize": min(max_results, 100),
            "format": "json",
        }
        try:
            r = self.session.get(self.BASE, params=params, timeout=15)
            data = r.json()
            studies = data.get("studies", [])
            return [self._parse(s) for s in studies]
        except Exception:
            return []

    @staticmethod
    def _parse(study: dict) -> dict:
        ps  = study.get("protocolSection", {})
        idf = ps.get("identificationModule", {})
        std = ps.get("statusModule",         {})
        des = ps.get("designModule",         {})
        cnd = ps.get("conditionsModule",     {})
        enr = des.get("enrollmentInfo",      {})

        # Primary completion date
        pcd = std.get("primaryCompletionDateStruct", {}).get("date", "")
        try:
            pcd_dt = pd.Timestamp(pcd)
        except Exception:
            pcd_dt = pd.NaT

        phases = des.get("phases", [])
        phase  = phases[0] if phases else "UNKNOWN"
        # Clean up phase string
        phase  = phase.replace("PHASE","Phase ").replace("_"," ").strip()

        return {
            "nct_id":          idf.get("nctId", ""),
            "title":           idf.get("briefTitle", "")[:100],
            "phase":           phase,
            "status":          std.get("overallStatus", ""),
            "sponsor":         ps.get("sponsorCollaboratorsModule",{}
                                      ).get("leadSponsor",{}).get("name",""),
            "conditions":      ", ".join(cnd.get("conditions", [])[:3]),
            "completion_date": pcd_dt,
            "enrollment":      enr.get("count", None),
        }


# ── Company name → search string mapping ─────────────────────
# ClinicalTrials uses legal company names, not tickers
CT_SEARCH_NAMES = {
    "VRTX": "Vertex Pharmaceuticals",
    "REGN": "Regeneron Pharmaceuticals",
    "ALNY": "Alnylam Pharmaceuticals",
    "BIIB": "Biogen",
    "BMRN": "BioMarin",
    "MRNA": "Moderna",
    "EXEL": "Exelixis",
    "INCY": "Incyte",
    "FOLD": "Amicus Therapeutics",
    "PTCT": "PTC Therapeutics",
    "RARE": "Ultragenyx",
    "ACAD": "ACADIA",
    "IONS": "Ionis Pharmaceuticals",
    "SRPT": "Sarepta",
    "BLUE": "bluebird bio",
    "CRSP": "CRISPR Therapeutics",
    "NTLA": "Intellia Therapeutics",
    "KRTX": "Karuna Therapeutics",
    "SAGE": "Sage Therapeutics",
    "RYTM": "Rhythm Pharmaceuticals",
    "GILD": "Gilead Sciences",
    "AMGN": "Amgen",
    "MRK":  "Merck Sharp",
    "PFE":  "Pfizer",
}

print("🔬  ClinicalTrials.gov Pipeline Tracker")
print(f"    Querying {len(CT_SEARCH_NAMES)} companies …\n")

ct_api    = ClinicalTrialsAPI()
ct_trials = []

for ticker, name in tqdm(CT_SEARCH_NAMES.items(), desc="  CT.gov queries", ncols=72):
    trials = ct_api.search_by_sponsor(name, max_results=30)
    for t in trials:
        t["ticker"] = ticker
    ct_trials.extend(trials)
    time.sleep(0.3)   # polite delay

ct_df = pd.DataFrame(ct_trials)

if not ct_df.empty:
    ct_df = ct_df.dropna(subset=["nct_id"])
    ct_df = ct_df[ct_df["phase"].str.contains("2|3|4", na=False)]   # Phase 2+
    ct_df.to_parquet(DATA_DIR / "clinical_trials.parquet", index=False)
    ct_df.to_csv(DATA_DIR     / "clinical_trials.csv",     index=False)

    print(f"\n✅  ClinicalTrials.gov pull complete")
    print(f"   Total trials (Phase 2+)  : {len(ct_df)}")
    print(f"   Unique companies         : {ct_df['ticker'].nunique()}")
    print("\n   Phase breakdown:")
    print(ct_df["phase"].value_counts().to_string())
    print("\n   Status breakdown:")
    print(ct_df["status"].value_counts().head(5).to_string())

    # Upcoming completions
    today = pd.Timestamp.today()
    upcoming = ct_df[
        (ct_df["completion_date"] >= today) &
        (ct_df["completion_date"] <= today + pd.Timedelta(days=CFG["upcoming_days"]))
    ].sort_values("completion_date")
    print(f"\n   Trials completing in next {CFG['upcoming_days']} days: {len(upcoming)}")
    if len(upcoming) > 0:
        print("\n   Next 10 upcoming trials:")
        display(upcoming[["ticker","phase","title","conditions","completion_date","status"]]
                .head(10).style.set_properties(**{"font-size":"10px"}))
else:
    print("  ⚠️  No trial data returned — check internet connection or API availability")
    ct_df = pd.DataFrame()


## Cell 5 — Event Classification & Database Construction
*Builds the clean event database used in all subsequent analyses. Applies additional filters and extracts drug names using regex.*

In [ ]:
# ============================================================
# CELL 5 — Event classification, enrichment & database build
# ============================================================

# Load raw events (from Cell 3, or use pre-cached if Cell 3 was skipped)
raw_path = DATA_DIR / "raw_events.parquet"
if raw_path.exists():
    events_df = pd.read_parquet(raw_path)
    print(f"  📂  Loaded {len(events_df)} raw events from cache")
else:
    print("  ⚠️  No raw_events.parquet found — run Cell 3 first")
    print("  💡  Using synthetic demo dataset for illustration …")

    # ── Synthetic demo dataset ─────────────────────────────────
    # Realistic events for illustration when live scraping hasn't run yet
    # Based on publicly announced clinical results 2020-2024
    DEMO_EVENTS = [
        # VRTX — Cystic fibrosis
        {"ticker":"VRTX","company":"Vertex Pharmaceuticals","date":"2021-06-21","event_type":"phase3_success","headline":"Vertex reports positive Phase 3 results for VX-445/tezacaftor/ivacaftor in CF patients"},
        {"ticker":"VRTX","company":"Vertex Pharmaceuticals","date":"2021-10-12","event_type":"fda_approval","headline":"FDA approves Trikafta label expansion for children ages 6-11 with CF"},
        # REGN
        {"ticker":"REGN","company":"Regeneron","date":"2022-05-23","event_type":"phase3_success","headline":"Dupixent Phase 3 trial met primary endpoint in COPD patients"},
        {"ticker":"REGN","company":"Regeneron","date":"2022-09-28","event_type":"fda_approval","headline":"FDA approves Dupixent for atopic dermatitis in children 6 months to 5 years"},
        # ALNY
        {"ticker":"ALNY","company":"Alnylam","date":"2021-08-12","event_type":"phase3_success","headline":"Inclisiran Phase 3 data showed 50% LDL reduction meeting primary endpoint"},
        {"ticker":"ALNY","company":"Alnylam","date":"2022-11-22","event_type":"fda_approval","headline":"FDA approves Alnylam inclisiran NDA for cardiovascular risk reduction"},
        # BIIB
        {"ticker":"BIIB","company":"Biogen","date":"2021-06-07","event_type":"fda_approval","headline":"FDA grants accelerated approval to Biogen aducanumab for Alzheimer's disease"},
        {"ticker":"BIIB","company":"Biogen","date":"2020-12-30","event_type":"phase3_failure","headline":"Biogen Phase 3 study did not meet primary endpoint for ALS treatment"},
        # MRNA
        {"ticker":"MRNA","company":"Moderna","date":"2021-08-23","event_type":"fda_approval","headline":"FDA grants full approval to Moderna COVID-19 vaccine for adults"},
        {"ticker":"MRNA","company":"Moderna","date":"2023-05-15","event_type":"phase3_success","headline":"Moderna mRNA-1345 RSV vaccine Phase 3 met primary endpoint in older adults"},
        # BMRN
        {"ticker":"BMRN","company":"BioMarin","date":"2020-08-19","event_type":"fda_rejection","headline":"BioMarin receives complete response letter for Roctavian gene therapy BLA"},
        {"ticker":"BMRN","company":"BioMarin","date":"2022-08-31","event_type":"fda_approval","headline":"FDA approves Roctavian for hemophilia A"},
        # SRPT
        {"ticker":"SRPT","company":"Sarepta","date":"2023-06-22","event_type":"fda_approval","headline":"FDA approves Sarepta Elevidys gene therapy for Duchenne muscular dystrophy"},
        {"ticker":"SRPT","company":"Sarepta","date":"2021-02-25","event_type":"phase3_success","headline":"Sarepta SRP-9001 Phase 3 met primary endpoint for functional improvement"},
        # SAGE
        {"ticker":"SAGE","company":"Sage Therapeutics","date":"2021-08-11","event_type":"phase3_failure","headline":"Sage Therapeutics LANDSCAPE trial did not meet primary endpoint for depression"},
        {"ticker":"SAGE","company":"Sage Therapeutics","date":"2023-06-29","event_type":"fda_approval","headline":"FDA approves Sage zuranolone for postpartum depression"},
        # CRSP
        {"ticker":"CRSP","company":"CRISPR Therapeutics","date":"2023-12-08","event_type":"fda_approval","headline":"FDA approves Casgevy first CRISPR gene editing therapy for sickle cell disease"},
        {"ticker":"CRSP","company":"CRISPR Therapeutics","date":"2021-12-14","event_type":"phase3_success","headline":"CRISPR CTX001 Phase 3 data showed complete resolution of vaso-occlusive crises"},
        # ACAD
        {"ticker":"ACAD","company":"ACADIA","date":"2022-08-30","event_type":"fda_rejection","headline":"FDA issues complete response letter for ACADIA pimavanserin Alzheimer dementia"},
        {"ticker":"ACAD","company":"ACADIA","date":"2023-06-01","event_type":"fda_approval","headline":"FDA approves ACADIA Daybue for Rett syndrome"},
        # RARE
        {"ticker":"RARE","company":"Ultragenyx","date":"2021-06-18","event_type":"phase3_success","headline":"Ultragenyx UX007 Phase 3 met primary endpoint in GLUT1 deficiency syndrome"},
        {"ticker":"RARE","company":"Ultragenyx","date":"2022-11-16","event_type":"fda_approval","headline":"FDA grants approval to Ultragenyx gene therapy for GSD Ia"},
        # INCY
        {"ticker":"INCY","company":"Incyte","date":"2022-09-16","event_type":"phase3_failure","headline":"Incyte parsaclisib failed to meet primary PFS endpoint in Phase 3 CITADEL-306"},
        {"ticker":"INCY","company":"Incyte","date":"2023-01-23","event_type":"fda_approval","headline":"FDA approves Incyte ruxolitinib cream for atopic dermatitis"},
        # EXEL
        {"ticker":"EXEL","company":"Exelixis","date":"2021-07-30","event_type":"phase3_success","headline":"Cabozantinib COSMIC-311 Phase 3 met PFS primary endpoint in radioiodine-refractory thyroid"},
        # Additional events for statistical power
        {"ticker":"NTLA","company":"Intellia","date":"2021-06-26","event_type":"phase1_data","headline":"Intellia NTLA-2001 Phase 1 data showed 87% reduction in TTR levels — proof of concept"},
        {"ticker":"BLUE","company":"bluebird bio","date":"2022-03-15","event_type":"fda_rejection","headline":"bluebird bio betibeglogene CRL received for manufacturing concerns"},
        {"ticker":"BLUE","company":"bluebird bio","date":"2022-08-05","event_type":"fda_approval","headline":"FDA approves bluebird bio Zynteglo gene therapy for beta-thalassemia"},
        {"ticker":"IONS","company":"Ionis","date":"2021-03-05","event_type":"phase3_success","headline":"Ionis eplontersen Phase 3 NEURO-TTRansform met primary endpoint"},
        {"ticker":"GILD","company":"Gilead","date":"2021-04-01","event_type":"phase3_failure","headline":"Gilead filgotinib Phase 3 SELECTION failed to show superiority over placebo"},
        {"ticker":"PTCT","company":"PTC Therapeutics","date":"2021-02-26","event_type":"phase3_failure","headline":"PTC Therapeutics emflaza Phase 3 SPLINTR did not meet primary endpoint"},
        {"ticker":"KRTX","company":"Karuna","date":"2023-05-11","event_type":"phase3_success","headline":"Karuna KarXT Phase 3 EMERGENT-4 met primary endpoint in schizophrenia"},
        {"ticker":"KRTX","company":"Karuna","date":"2024-09-26","event_type":"fda_approval","headline":"FDA approves Karuna Cobenfy first muscarinic mechanism antipsychotic"},
        {"ticker":"FOLD","company":"Amicus","date":"2021-05-28","event_type":"phase3_success","headline":"Amicus AT-GAA Phase 3 PROPEL met primary endpoint in Pompe disease"},
        {"ticker":"FOLD","company":"Amicus","date":"2022-08-05","event_type":"fda_approval","headline":"FDA approves Pombiliti for late-onset Pompe disease"},
        {"ticker":"RYTM","company":"Rhythm","date":"2020-11-27","event_type":"fda_approval","headline":"FDA approves Rhythm setmelanotide for rare genetic obesity disorders"},
        {"ticker":"ARQT","company":"Arcutis","date":"2022-07-20","event_type":"fda_approval","headline":"FDA approves Arcutis roflumilast cream for plaque psoriasis in adults and children"},
        {"ticker":"IMVT","company":"Immunovant","date":"2022-02-28","event_type":"phase2_success","headline":"Immunovant IMVT-1401 Phase 2 positive — 75% IgG reduction in myasthenia gravis"},
        {"ticker":"PRTA","company":"Prothena","date":"2023-05-01","event_type":"phase3_failure","headline":"Prothena prasinezumab Phase 2b did not meet primary endpoint in Parkinson disease"},
    ]

    events_df = pd.DataFrame(DEMO_EVENTS)
    events_df["date"] = pd.to_datetime(events_df["date"])
    events_df["stage"]    = events_df["event_type"].map(EVENT_STAGE)
    events_df["polarity"] = events_df["event_type"].map(EVENT_POLARITY)
    events_df["doc_url"]  = ""
    events_df["cik"]      = ""
    print(f"  ✅  Demo dataset loaded: {len(events_df)} events")

# ── Enrichment ─────────────────────────────────────────────────
events_df["date"] = pd.to_datetime(events_df["date"])
events_df["year"] = events_df["date"].dt.year

# Extract drug name from headline using regex
def extract_drug_name(headline: str) -> str:
    """
    Extracts potential drug/compound name from headline.
    Looks for: CamelCase names, NDA numbers, PHASE patterns.
    """
    # Pattern: capitalized word 4-15 chars that looks like a drug name
    matches = re.findall(r'\b[A-Z][a-z]{3,14}(?:[a-z]{2,8})?\b', str(headline))
    # Filter out common non-drug capitalized words
    stop = {"Phase","Trial","Study","FDA","Results","Data","Primary","Endpoint",
            "Approval","Announces","Reported","Positive","Negative","Patients",
            "Treatment","Disease","Clinical","Syndrome","Company","Reports",
            "Monday","Tuesday","Wednesday","Thursday","Friday"}
    candidates = [m for m in matches if m not in stop]
    return candidates[0] if candidates else ""

events_df["drug_name"] = events_df["headline"].apply(extract_drug_name)

# Final clean
events_df = events_df.sort_values("date").reset_index(drop=True)
events_df.to_parquet(DATA_DIR / "events_clean.parquet", index=False)

print(f"\n✅  Event database built")
print(f"   Total events   : {len(events_df)}")
print(f"   Tickers covered: {events_df['ticker'].nunique()}")
print(f"   Date range     : {events_df['date'].min().date()} → {events_df['date'].max().date()}")
print("\n   Event breakdown:")
breakdown = events_df.groupby(["stage","polarity"]).size().unstack(fill_value=0)
print(breakdown.to_string())

# Quick timeline chart
fig, ax = plt.subplots(figsize=(13, 4))
colors_map = {"positive": C["success"], "negative": C["failure"], "neutral": C["neutral"]}
for _, row in events_df.iterrows():
    ax.scatter(row["date"], row["stage"],
               color=colors_map[row["polarity"]], s=55, alpha=0.7, zorder=3)
ax.set_title("Clinical event timeline — color = outcome polarity")
ax.set_xlabel("Date")
ax.set_ylabel("Stage")
pos_p = mpatches.Patch(color=C["success"], label="Positive")
neg_p = mpatches.Patch(color=C["failure"], label="Negative")
neu_p = mpatches.Patch(color=C["neutral"], label="Neutral")
ax.legend(handles=[pos_p, neg_p, neu_p], loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(DATA_DIR / "event_timeline.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 6 — Price Data & Abnormal Return Computation
*Downloads price history for every stock and benchmark. Computes abnormal returns using the market model (OLS estimation window). This is standard event study methodology (MacKinlay 1997).*

In [ ]:
# ============================================================
# CELL 6 — Price data ingestion & abnormal return computation
# ============================================================

class EventStudy:
    """
    Standard event study methodology (MacKinlay 1997):

    1. Estimation window: [-252, -30] days before event
       → Estimate normal return model: R_i = α + β * R_m + ε

    2. Event window: [-pre, +post] days around event
       → Compute abnormal return: AR_t = R_it - (α̂ + β̂ * R_mt)

    3. Cumulative abnormal return: CAR[t1, t2] = Σ AR_t

    The XBI (biotech ETF) is used as the benchmark rather than SPY
    because biotech stocks move in herds — a sector-adjusted benchmark
    removes the noise of general biotech sentiment and isolates
    company-specific news (the clinical trial outcome).
    """

    def __init__(self, events: pd.DataFrame, cfg: dict):
        self.events = events
        self.cfg    = cfg
        self.prices = {}     # ticker → pd.Series (daily adjusted close)
        self.bench  = None   # benchmark daily returns

    # ── Download prices ──────────────────────────────────────
    def download_prices(self, start: str = "2017-01-01") -> None:
        print("  ⬇️   Downloading price data …")
        all_tickers = list(self.events["ticker"].unique()) + [self.cfg["benchmark"]]

        raw = yf.download(all_tickers, start=start,
                          auto_adjust=True, progress=False)
        closes = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw

        self.bench = closes[self.cfg["benchmark"]].pct_change().dropna()
        for ticker in self.events["ticker"].unique():
            if ticker in closes.columns:
                s = closes[ticker].dropna()
                if len(s) > 250:
                    self.prices[ticker] = s
        print(f"  ✅  Prices loaded for {len(self.prices)} tickers")

    # ── Compute abnormal return for one event ────────────────
    def compute_ar(self, ticker: str, event_date: pd.Timestamp) -> dict | None:
        """
        Returns dict with:
          - dates    : array of dates in event window
          - days     : array of relative days [-pre, ..., +post]
          - ar       : abnormal return per day
          - car      : cumulative abnormal return per day
          - alpha, beta: market model parameters
          - event_day_ar: AR on day 0 (the announcement day)
        """
        if ticker not in self.prices:
            return None

        price = self.prices[ticker]
        ret   = price.pct_change().dropna()
        bench = self.bench

        # Align on common index
        common = ret.index.intersection(bench.index)
        ret    = ret.reindex(common)
        bench  = bench.reindex(common)

        # Find event date position
        trading_dates = common
        if event_date not in trading_dates:
            # Use nearest trading day
            diffs = abs(trading_dates - event_date)
            nearest = trading_dates[diffs.argmin()]
            if abs(nearest - event_date).days > 5:
                return None
            event_date = nearest

        ev_idx = trading_dates.get_loc(event_date)

        # Estimation window indices
        est_start = ev_idx + self.cfg["estimation_start"]
        est_end   = ev_idx + self.cfg["estimation_end"]
        if est_start < 0:
            return None

        # Event window indices
        pre  = self.cfg["pre_window"]
        post = self.cfg["post_window"]
        win_start = max(0, ev_idx - pre)
        win_end   = min(len(trading_dates) - 1, ev_idx + post)

        if win_end - win_start < 30:   # too short
            return None

        # ── Market model estimation ──────────────────────────
        est_ret   = ret.iloc[est_start:est_end]
        est_bench = bench.iloc[est_start:est_end]

        if len(est_ret) < 60:          # need enough estimation obs
            return None

        X = sm.add_constant(est_bench.values)
        try:
            ols   = sm.OLS(est_ret.values, X).fit()
            alpha = ols.params[0]
            beta  = ols.params[1]
        except Exception:
            alpha, beta = 0.0, 1.0

        # ── Event window abnormal returns ────────────────────
        ev_ret   = ret.iloc[win_start:win_end+1]
        ev_bench = bench.iloc[win_start:win_end+1]
        ev_dates = trading_dates[win_start:win_end+1]

        expected = alpha + beta * ev_bench.values
        ar       = ev_ret.values - expected
        car      = np.cumsum(ar)
        days     = np.arange(len(ar)) - (ev_idx - win_start)  # relative days

        return {
            "ticker":       ticker,
            "event_date":   event_date,
            "days":         days,
            "dates":        ev_dates,
            "ar":           ar,
            "car":          car,
            "alpha":        alpha,
            "beta":         beta,
            "event_day_ar": ar[ev_idx - win_start] if (ev_idx - win_start) < len(ar) else np.nan,
            "car_0_5":      car[min(ev_idx - win_start + 5, len(car)-1)]  - car[ev_idx - win_start],
            "car_0_21":     car[min(ev_idx - win_start + 21, len(car)-1)] - car[ev_idx - win_start],
            "car_m5_0":     car[ev_idx - win_start] - car[max(0, ev_idx - win_start - 5)],
        }

    # ── Compute for entire event database ────────────────────
    def run_all(self) -> pd.DataFrame:
        print("  Computing abnormal returns for all events …")
        results = []
        for _, row in tqdm(self.events.iterrows(),
                           total=len(self.events), desc="  Events", ncols=72):
            ar_dict = self.compute_ar(row["ticker"], row["date"])
            if ar_dict is None:
                continue
            ar_dict.update({
                "event_type": row["event_type"],
                "stage":      row["stage"],
                "polarity":   row["polarity"],
                "company":    row["company"],
                "headline":   row["headline"],
                "drug_name":  row.get("drug_name",""),
                "year":       row["date"].year,
            })
            results.append(ar_dict)

        df = pd.DataFrame(results)
        print(f"  ✅  {len(df)} events successfully processed")
        return df


# ── Run ──────────────────────────────────────────────────────
print("📈  Cell 6: Price Data & Abnormal Return Computation")
events_df = pd.read_parquet(DATA_DIR / "events_clean.parquet")

es      = EventStudy(events_df, CFG)
es.download_prices(start="2017-01-01")
ar_df   = es.run_all()

# Save serialisable version (drop array columns for parquet)
scalar_cols = ["ticker","company","event_type","stage","polarity","headline",
               "drug_name","year","event_date","event_day_ar","car_0_5","car_0_21","car_m5_0","alpha","beta"]
ar_scalar   = ar_df[scalar_cols].copy()
ar_scalar.to_parquet(DATA_DIR / "ar_results.parquet", index=False)

print("\n   Summary statistics — event-day AR by polarity:")
print(ar_scalar.groupby("polarity")["event_day_ar"]
      .agg(["mean","median","std","count"])
      .round(4).to_string())


## Cell 7 — Event Study Analysis & Core Results
*The main findings. Computes average cumulative abnormal returns (ACAR) for each event type, tests statistical significance, and produces the key charts that would appear in a sell-side research note.*

In [ ]:
# ============================================================
# CELL 7 — Core event study analysis & visualizations
# ============================================================

ar_scalar = pd.read_parquet(DATA_DIR / "ar_results.parquet")

# ── Helper: interpolate average CAR path ─────────────────────
def avg_car_path(subset_df, ar_full_df, days_range=(-30, 30)):
    """
    Reconstruct average cumulative abnormal return path
    from individual event ar/car arrays.
    """
    day_grid = np.arange(days_range[0], days_range[1]+1)
    car_matrix = []

    idx = subset_df.index.tolist()
    for i in idx:
        row = ar_full_df.loc[i]
        days = row["days"]
        car  = row["car"]
        # Interpolate onto day_grid
        interp = np.full(len(day_grid), np.nan)
        for j, d in enumerate(days):
            if days_range[0] <= d <= days_range[1]:
                g_idx = d - days_range[0]
                if 0 <= g_idx < len(day_grid):
                    interp[g_idx] = car[j]
        car_matrix.append(interp)

    mat = np.array(car_matrix)
    mean_car = np.nanmean(mat, axis=0)
    se_car   = np.nanstd(mat, axis=0) / np.sqrt(np.sum(~np.isnan(mat), axis=0).clip(1))
    return day_grid, mean_car, se_car

# ── Figure 1: Average CAR by event type ──────────────────────
STAGE_GROUPS = {
    "FDA Approval":   ar_df[(ar_df["stage"]=="FDA")      & (ar_df["polarity"]=="positive")],
    "FDA Rejection":  ar_df[(ar_df["stage"]=="FDA")      & (ar_df["polarity"]=="negative")],
    "Phase 3 ✓":     ar_df[(ar_df["stage"]=="Phase 3")  & (ar_df["polarity"]=="positive")],
    "Phase 3 ✗":     ar_df[(ar_df["stage"]=="Phase 3")  & (ar_df["polarity"]=="negative")],
    "Phase 2 ✓":     ar_df[(ar_df["stage"]=="Phase 2")  & (ar_df["polarity"]=="positive")],
    "Phase 2 ✗":     ar_df[(ar_df["stage"]=="Phase 2")  & (ar_df["polarity"]=="negative")],
}
GROUP_COLORS = {
    "FDA Approval":  C["success"],
    "FDA Rejection": C["failure"],
    "Phase 3 ✓":    C["phase3"],
    "Phase 3 ✗":    "#C4327B",
    "Phase 2 ✓":    C["phase2"],
    "Phase 2 ✗":    "#E28A30",
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
DAYS_RANGE = (-20, 30)

for ax, (label, subset) in zip(axes, STAGE_GROUPS.items()):
    if len(subset) == 0:
        ax.text(0.5, 0.5, f"No {label}\nevents in dataset",
                ha="center", va="center", transform=ax.transAxes,
                color=C["neutral"])
        ax.set_title(label)
        continue

    day_grid, mean_car, se_car = avg_car_path(subset, ar_df, DAYS_RANGE)

    ax.fill_between(day_grid, (mean_car-1.96*se_car)*100,
                    (mean_car+1.96*se_car)*100,
                    alpha=0.15, color=GROUP_COLORS[label])
    ax.plot(day_grid, mean_car*100, lw=2.2,
            color=GROUP_COLORS[label], label=f"ACAR (n={len(subset)})")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(0, color="black", lw=1.2, ls="--", alpha=0.6)

    # Annotate day-0 CAR
    d0_idx = np.where(day_grid == 0)[0]
    if len(d0_idx):
        d0_car = mean_car[d0_idx[0]]
        ax.annotate(f"Day 0: {d0_car*100:+.1f}%",
                    xy=(0, d0_car*100), xytext=(5, d0_car*100 + 1),
                    fontsize=8.5, color=GROUP_COLORS[label])

    ax.set_title(f"{label}  (n = {len(subset)})")
    ax.set_xlabel("Trading days relative to event")
    ax.set_ylabel("Avg CAR (%)")
    ax.legend(fontsize=8)

plt.suptitle("Average Cumulative Abnormal Return (ACAR) by Clinical Event Type "
             "Benchmark: XBI (SPDR S&P Biotech ETF)  |  Shaded = 95% CI",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / "acar_by_event.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 2: Day-0 return distribution by event type ────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, stage in zip(axes, ["Phase 3", "FDA"]):
    pos = ar_scalar[(ar_scalar["stage"]==stage)&(ar_scalar["polarity"]=="positive")]["event_day_ar"].dropna()*100
    neg = ar_scalar[(ar_scalar["stage"]==stage)&(ar_scalar["polarity"]=="negative")]["event_day_ar"].dropna()*100

    if len(pos) > 0:
        ax.hist(pos, bins=20, alpha=0.65, color=C["success"],
                label=f"Success (n={len(pos)}, μ={pos.mean():.1f}%)", edgecolor="white")
    if len(neg) > 0:
        ax.hist(neg, bins=20, alpha=0.65, color=C["failure"],
                label=f"Failure (n={len(neg)}, μ={neg.mean():.1f}%)", edgecolor="white")

    ax.axvline(0, color="black", lw=1)
    ax.set_title(f"{stage} event — day-0 abnormal return distribution")
    ax.set_xlabel("Day-0 abnormal return (%)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(DATA_DIR / "day0_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Statistical significance table ──────────────────────────
print("\n📊  Statistical significance — day-0 abnormal returns")
print("="*68)
print(f"  {'Event Type':<22} {'n':>4} {'Mean AR':>9} {'Median':>9} {'Std':>9} {'t-stat':>8} {'p-val':>8}")
print("  " + "-"*66)

sig_rows = []
for label, subset in STAGE_GROUPS.items():
    d0 = ar_scalar.loc[subset.index, "event_day_ar"].dropna() if len(subset) > 0 else pd.Series(dtype=float)
    if len(d0) < 3:
        continue
    t, p = stats.ttest_1samp(d0, 0)
    sig  = "***" if p<0.01 else ("**" if p<0.05 else ("*" if p<0.1 else ""))
    print(f"  {label:<22} {len(d0):>4} {d0.mean()*100:>8.1f}% {d0.median()*100:>8.1f}% "
          f"{d0.std()*100:>8.1f}% {t:>7.2f}  {p:>6.4f} {sig}")
    sig_rows.append({"event":label,"n":len(d0),"mean_ar":d0.mean(),"pvalue":p})

print("  " + "-"*66)
print("  * p<0.10  ** p<0.05  *** p<0.01")
print("  Hypothesis: H0: mean abnormal return = 0")
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(DATA_DIR / "significance_table.csv", index=False)


## Cell 8 — Phase 2 → Phase 3 Pricing Analysis
*The core insight that defines great healthcare equity research: how much of a Phase 3 success is already priced in at Phase 2? This is exactly the question clients pay healthcare analysts to answer.*

In [ ]:
# ============================================================
# CELL 8 — Phase 2 → Phase 3 pricing & signal persistence
# ============================================================

ar_scalar = pd.read_parquet(DATA_DIR / "ar_results.parquet")

print("🧪  Phase 2 → Phase 3 Pricing Analysis")
print("="*60)
print("""
KEY QUESTION FOR HEALTHCARE ER:
  If a company announces a Phase 2 SUCCESS,
  how much of the subsequent Phase 3 SUCCESS
  is already priced in by the time Phase 3 data reads out?

  This is the 'Phase 2 → Phase 3 pricing efficiency' question.
  Great biotech investors — and great healthcare analysts — can
  answer this quantitatively.
""")

# ── Figure 3: CAR persistence after each event type ──────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

windows = [
    ("car_m5_0",  "Pre-event drift\n(-5 to 0 days)"),
    ("event_day_ar", "Announcement day\n(day 0)"),
    ("car_0_21",  "Post-event drift\n(day 0 to +21)"),
]

for ax, (col, label) in zip(axes, windows):
    data_plot = []
    labels    = []
    clrs      = []

    for stage in ["Phase 2", "Phase 3", "FDA"]:
        for polarity, sym in [("positive","✓"), ("negative","✗")]:
            sub = ar_scalar[(ar_scalar["stage"]==stage)&(ar_scalar["polarity"]==polarity)][col].dropna()*100
            if len(sub) < 2:
                continue
            data_plot.append(sub.values)
            labels.append(f"{stage} {sym}")
            clrs.append(C["success"] if polarity=="positive" else C["failure"])

    if not data_plot:
        ax.set_visible(False)
        continue

    bp = ax.boxplot(data_plot, patch_artist=True, notch=False,
                    medianprops={"color":"black","lw":2},
                    flierprops={"marker":"o","markersize":4,"alpha":0.4})
    for patch, c in zip(bp["boxes"], clrs):
        patch.set_facecolor(c)
        patch.set_alpha(0.65)

    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(label)
    ax.set_ylabel("Abnormal return (%)")

plt.suptitle("Abnormal return timing: pre-event leak, announcement, and post-drift", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "car_timing.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 4: Phase 2 vs Phase 3 returns comparison ──────────
fig, ax = plt.subplots(figsize=(10, 5))

categories = {
    "Phase 2\nSuccess": ar_scalar[(ar_scalar["stage"]=="Phase 2")&(ar_scalar["polarity"]=="positive")]["event_day_ar"].dropna()*100,
    "Phase 2\nFailure": ar_scalar[(ar_scalar["stage"]=="Phase 2")&(ar_scalar["polarity"]=="negative")]["event_day_ar"].dropna()*100,
    "Phase 3\nSuccess": ar_scalar[(ar_scalar["stage"]=="Phase 3")&(ar_scalar["polarity"]=="positive")]["event_day_ar"].dropna()*100,
    "Phase 3\nFailure": ar_scalar[(ar_scalar["stage"]=="Phase 3")&(ar_scalar["polarity"]=="negative")]["event_day_ar"].dropna()*100,
    "FDA\nApproval":    ar_scalar[(ar_scalar["stage"]=="FDA")    &(ar_scalar["polarity"]=="positive")]["event_day_ar"].dropna()*100,
    "FDA\nRejection":   ar_scalar[(ar_scalar["stage"]=="FDA")    &(ar_scalar["polarity"]=="negative")]["event_day_ar"].dropna()*100,
}

cat_means = {k: v.mean() for k, v in categories.items() if len(v) > 0}
cat_se    = {k: v.sem()  for k, v in categories.items() if len(v) > 0}
cat_n     = {k: len(v)   for k, v in categories.items() if len(v) > 0}

keys   = list(cat_means.keys())
means  = [cat_means[k] for k in keys]
sems   = [cat_se[k]    for k in keys]
ns     = [cat_n[k]     for k in keys]
bar_c  = [C["success"] if m>0 else C["failure"] for m in means]

bars = ax.bar(keys, means, yerr=sems, color=bar_c, alpha=0.8,
              edgecolor="white", capsize=5, width=0.6)
ax.axhline(0, color="black", lw=0.8)
for bar, m, n in zip(bars, means, ns):
    y_off = 0.5 if m >= 0 else -1.5
    ax.text(bar.get_x() + bar.get_width()/2, m + y_off,
            f"{m:+.1f}%\n(n={n})", ha="center", va="bottom", fontsize=8.5)

ax.set_title("Mean day-0 abnormal return by clinical event type\n"
             "Error bars = standard error of mean")
ax.set_ylabel("Mean day-0 abnormal return (%)")
plt.tight_layout()
plt.savefig(DATA_DIR / "mean_ar_by_type.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Phase 2→3 pricing efficiency analysis ────────────────────
ph2_success  = ar_scalar[(ar_scalar["stage"]=="Phase 2") & (ar_scalar["polarity"]=="positive")]
ph3_success  = ar_scalar[(ar_scalar["stage"]=="Phase 3") & (ar_scalar["polarity"]=="positive")]
fda_approval = ar_scalar[(ar_scalar["stage"]=="FDA")     & (ar_scalar["polarity"]=="positive")]
ph2_fail     = ar_scalar[(ar_scalar["stage"]=="Phase 2") & (ar_scalar["polarity"]=="negative")]
ph3_fail     = ar_scalar[(ar_scalar["stage"]=="Phase 3") & (ar_scalar["polarity"]=="negative")]
fda_crl      = ar_scalar[(ar_scalar["stage"]=="FDA")     & (ar_scalar["polarity"]=="negative")]

def safe_mean(df, col): return df[col].dropna().mean()*100 if len(df)>0 else 0
def safe_n(df): return len(df)

print("\n📊  Key findings — Clinical event return summary")
print("="*62)
print(f"  {'Event Type':<25} {'Day-0 AR':>10} {'0→+21 CAR':>10} {'n':>5}")
print("  " + "-"*60)
rows_ = [
    ("Phase 2 success",  ph2_success),
    ("Phase 2 failure",  ph2_fail),
    ("Phase 3 success",  ph3_success),
    ("Phase 3 failure",  ph3_fail),
    ("FDA approval",     fda_approval),
    ("FDA rejection/CRL",fda_crl),
]
for name, sub in rows_:
    d0 = safe_mean(sub, "event_day_ar")
    c21= safe_mean(sub, "car_0_21")
    n  = safe_n(sub)
    flag = " ← LARGEST NEGATIVE" if d0 < -15 else (" ← NOTE: small positive" if name.startswith("FDA approval") and d0 < 5 else "")
    print(f"  {name:<25} {d0:>+9.1f}% {c21:>+9.1f}%  {n:>4}{flag}")
print("  " + "-"*60)

if safe_n(ph3_success) > 0 and safe_n(ph2_success) > 0:
    ratio = safe_mean(ph3_success,"event_day_ar") / safe_mean(ph2_success,"event_day_ar") if safe_mean(ph2_success,"event_day_ar") != 0 else 0
    print(f"\n  Phase 3 success AR / Phase 2 success AR = {ratio:.2f}x")
    print("  → Interpretation: If Phase 3 AR is SMALLER than Phase 2 AR,")
    print("    the market partially priced in the Phase 3 success at Phase 2.")
    print("    This is 'pricing efficiency' — key insight for risk-arb in biotech.")


## Cell 9 — Upcoming Catalyst Dashboard
*Forward-looking view: which companies in our universe have upcoming binary events? Built from ClinicalTrials.gov data. This is the live tool a healthcare analyst maintains for clients.*

In [ ]:
# ============================================================
# CELL 9 — Upcoming catalyst dashboard & individual case studies
# ============================================================

ct_path = DATA_DIR / "clinical_trials.parquet"

print("📅  Upcoming Catalyst Dashboard")

# ── Load ClinicalTrials data ──────────────────────────────────
if ct_path.exists():
    ct_df = pd.read_parquet(ct_path)
else:
    print("  ⚠️  No ClinicalTrials data — run Cell 4 first. Using placeholder.")
    ct_df = pd.DataFrame(columns=["ticker","phase","title","conditions",
                                   "completion_date","status","enrollment"])

today    = pd.Timestamp.today()
upcoming = pd.DataFrame()

if not ct_df.empty:
    ct_df["completion_date"] = pd.to_datetime(ct_df["completion_date"], errors="coerce")
    upcoming = ct_df[
        (ct_df["completion_date"] >= today) &
        (ct_df["completion_date"] <= today + pd.Timedelta(days=365)) &
        (ct_df["phase"].str.contains("2|3", na=False))
    ].copy()
    upcoming["days_until"] = (upcoming["completion_date"] - today).dt.days
    upcoming = upcoming.sort_values("days_until")

# ── Figure 5: Catalyst timeline ───────────────────────────────
fig, ax = plt.subplots(figsize=(13, max(5, min(len(upcoming)*0.35+2, 14))))

if len(upcoming) > 0:
    show_n = min(30, len(upcoming))
    up_show = upcoming.head(show_n).reset_index(drop=True)

    phase_colors = {
        "Phase 2": C["phase2"],
        "Phase 3": C["phase3"],
        "Phase 4": C["success"],
    }
    for i, row in up_show.iterrows():
        phase_key = "Phase 3" if "3" in str(row["phase"]) else (
                    "Phase 4" if "4" in str(row["phase"]) else "Phase 2")
        clr = phase_colors.get(phase_key, C["neutral"])
        ax.barh(i, row["days_until"], left=0, height=0.6,
                color=clr, alpha=0.8, edgecolor="white")
        ax.text(row["days_until"]+2, i,
                f"{row['ticker']} — {row['phase']}",
                va="center", ha="left", fontsize=8.5)

    ax.set_yticks(range(len(up_show)))
    title_wrap = [t[:50]+"…" if len(t)>50 else t for t in up_show["title"]]
    ax.set_yticklabels(title_wrap, fontsize=7.5)
    ax.set_xlabel("Days until primary completion date")
    ax.set_xlim(0, 400)
    ax.axvline(90,  color=C["failure"], lw=1, ls="--", alpha=0.6, label="90 days")
    ax.axvline(180, color=C["warning"], lw=1, ls="--", alpha=0.6, label="180 days")
    ax.axvline(365, color=C["neutral"], lw=1, ls="--", alpha=0.6, label="1 year")
    ax.legend(loc="lower right", fontsize=9)

    # Color legend
    for phase, clr in phase_colors.items():
        ax.scatter([], [], color=clr, s=60, label=phase)
    ax.legend(loc="lower right", fontsize=9)
    ax.set_title(f"Upcoming clinical trial completions — next 12 months\n"
                 f"(n={len(upcoming)} trials across {upcoming['ticker'].nunique()} companies)")
else:
    ax.text(0.5, 0.5, "No upcoming catalyst data available\nRun Cell 4 to pull ClinicalTrials.gov data",
            ha="center", va="center", transform=ax.transAxes, fontsize=12, color=C["neutral"])
    ax.set_title("Upcoming catalyst calendar — no data")

plt.tight_layout()
plt.savefig(DATA_DIR / "catalyst_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 6: Individual company deep-dive ───────────────────
print("\n🔬  Individual Event Deep-Dives")

ar_scalar = pd.read_parquet(DATA_DIR / "ar_results.parquet")
# Show top 6 most interesting events by absolute event_day_ar
top_events = pd.concat([
    ar_scalar.nlargest(3, "event_day_ar"),
    ar_scalar.nsmallest(3, "event_day_ar")
]).reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (_, ev_row) in zip(axes, top_events.iterrows()):
    # Find this event in ar_df (has full car array)
    matches = ar_df[
        (ar_df["ticker"] == ev_row["ticker"]) &
        (ar_df["event_date"] == pd.Timestamp(ev_row["event_date"]))
    ]
    if len(matches) == 0:
        ax.set_visible(False)
        continue

    row = matches.iloc[0]
    days, car = row["days"], row["car"]

    mask = (days >= -20) & (days <= 30)
    d_plot = days[mask]
    c_plot = car[mask]

    clr = C["success"] if ev_row["polarity"] == "positive" else C["failure"]
    ax.plot(d_plot, c_plot*100, lw=2, color=clr)
    ax.fill_between(d_plot, 0, c_plot*100, alpha=0.15, color=clr)
    ax.axhline(0, color="black", lw=0.7)
    ax.axvline(0, color="black", lw=1.2, ls="--", alpha=0.7)
    ax.set_title(f"{ev_row['ticker']} — {ev_row['event_type'].replace('_',' ').title()}\n"
                 f"{str(ev_row['event_date'])[:10]}  |  Day-0: {ev_row['event_day_ar']*100:+.1f}%",
                 fontsize=9)
    ax.set_xlabel("Days relative to event")
    ax.set_ylabel("CAR (%)")

plt.suptitle("Individual event deep-dives: top 3 gainers + top 3 losers (day-0 AR)", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "individual_events.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Catalyst monitoring table ─────────────────────────────────
print("\n📋  Catalyst Monitoring Table — next 90 days")
if len(upcoming) > 0:
    near_term = upcoming[upcoming["days_until"] <= 90].copy()
    near_term["completion_date"] = near_term["completion_date"].dt.strftime("%Y-%m-%d")
    if len(near_term) > 0:
        display(near_term[["ticker","phase","conditions","completion_date","days_until","enrollment"]]
                .rename(columns={"days_until":"days_until_completion"})
                .style.background_gradient(subset=["days_until_completion"],
                                            cmap="RdYlGn_r", vmin=0, vmax=90)
                .set_properties(**{"font-size":"11px"}))
    else:
        print("  No trials completing within 90 days in current dataset")
else:
    print("  No ClinicalTrials data — run Cell 4 to populate")
